In [1]:
!pip install -q langgraph langchain-mistralai langchain-core

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_mistralai import ChatMistralAI
from langgraph.checkpoint.memory import InMemorySaver
import os
from google.colab import userdata

In [3]:
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

llm = ChatMistralAI(model="mistral-small-latest")

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

# InMemorySaver = a CHECKPOINTER. It saves the graph's state after EVERY node
# runs, keyed by a "thread_id". Without a checkpointer, invoke() runs once
# and the state vanishes -- with one, every step is persisted, replayable,
# and resumable.
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
# thread_id is like a CONVERSATION ID -- all state for this run gets saved
# under "1". Every future call using config1 continues THIS thread's history.
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni',
 'explanation': 'The joke plays on the word *"boo"*—a common sound associated with ghosts—and combines it with *"pepperoni"* to create *"boo*-beroni."\n\nSince ghosts are often connected to spooky themes and the sound *"boo"* is a playful ghost sound, the punchline humorously suggests that ghosts would order a pizza with a ghostly twist, replacing the usual *"pepperoni"* with *"boo"* as a nod to their eerie nature.\n\nIt\'s a pun-based joke that mixes wordplay with a light-hearted ghost theme.'}

In [9]:
workflow.get_state(config1)
# returns the LATEST checkpoint for thread "1" -- topic, joke, explanation,
# plus metadata about which node ran last, etc.

StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word *"boo"*—a common sound associated with ghosts—and combines it with *"pepperoni"* to create *"boo*-beroni."\n\nSince ghosts are often connected to spooky themes and the sound *"boo"* is a playful ghost sound, the punchline humorously suggests that ghosts would order a pizza with a ghostly twist, replacing the usual *"pepperoni"* with *"boo"* as a nod to their eerie nature.\n\nIt\'s a pun-based joke that mixes wordplay with a light-hearted ghost theme.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-de9d-6e2e-8002-77cce97cf88c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T16:18:14.229122+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-d196-679c-8001-0427b19c02ad'}}, tasks=(), interrupts=())

In [10]:
list(workflow.get_state_history(config1))
# every INTERMEDIATE checkpoint for this thread, one per node execution --
# not just the final state. This is what makes "time travel" (below) possible.

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word *"boo"*—a common sound associated with ghosts—and combines it with *"pepperoni"* to create *"boo*-beroni."\n\nSince ghosts are often connected to spooky themes and the sound *"boo"* is a playful ghost sound, the punchline humorously suggests that ghosts would order a pizza with a ghostly twist, replacing the usual *"pepperoni"* with *"boo"* as a nod to their eerie nature.\n\nIt\'s a pun-based joke that mixes wordplay with a light-hearted ghost theme.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-de9d-6e2e-8002-77cce97cf88c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T16:18:14.229122+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-d196-679c-8001-0427b19c02ad'}}, tasks=(), interrupts=()),
 StateS

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'What do you call fake spaghetti? An impasta',
 'explanation': 'The joke *What do you call fake spaghetti? An impasta* plays on the word *"impasta"* sounding like *"imposter"* and *"pasta."*\n\nHere\'s the breakdown:\n- **"Spaghetti"** is a type of pasta.\n- **"Impasta"** sounds like *"imposter"* (a person pretending to be something they\'re not) + *"pasta."*\n- The humor comes from the pun, implying that fake spaghetti is called an *"impasta"* because it\'s an *imposter* of real pasta.\n\nIt’s a simple but effective pun that relies on wordplay and the listener recognizing the double meaning of *"impasta."*'}

In [12]:
workflow.get_state(config1)
# still shows the PIZZA joke -- threads are completely isolated from each
# other. This is the mechanism behind separate user conversations in a real
# chatbot: one thread_id per user/session, all backed by the same compiled graph.

StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word *"boo"*—a common sound associated with ghosts—and combines it with *"pepperoni"* to create *"boo*-beroni."\n\nSince ghosts are often connected to spooky themes and the sound *"boo"* is a playful ghost sound, the punchline humorously suggests that ghosts would order a pizza with a ghostly twist, replacing the usual *"pepperoni"* with *"boo"* as a nod to their eerie nature.\n\nIt\'s a pun-based joke that mixes wordplay with a light-hearted ghost theme.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-de9d-6e2e-8002-77cce97cf88c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T16:18:14.229122+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-d196-679c-8001-0427b19c02ad'}}, tasks=(), interrupts=())

In [13]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word *"boo"*—a common sound associated with ghosts—and combines it with *"pepperoni"* to create *"boo*-beroni."\n\nSince ghosts are often connected to spooky themes and the sound *"boo"* is a playful ghost sound, the punchline humorously suggests that ghosts would order a pizza with a ghostly twist, replacing the usual *"pepperoni"* with *"boo"* as a nod to their eerie nature.\n\nIt\'s a pun-based joke that mixes wordplay with a light-hearted ghost theme.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-de9d-6e2e-8002-77cce97cf88c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-23T16:18:14.229122+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-d196-679c-8001-0427b19c02ad'}}, tasks=(), interrupts=()),
 StateS

### Time Travel

In [15]:
# ⚠️ IMPORTANT: the checkpoint_id below is a REAL UUID from YOUR OWN run's
# history (Cell 12's output), NOT this literal string -- copy one from your
# actual get_state_history() output before running this.
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f19f0e3-d196-679c-8001-0427b19c02ad"}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f19f0e3-d196-679c-8001-0427b19c02ad'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-23T16:18:12.862926+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-cb0c-67a3-8000-d306094d6b60'}}, tasks=(PregelTask(id='809dc46f-3f90-51ff-a432-2e89c0546f32', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': 'The joke plays on the word *"boo"*—a common sound associated with ghosts—and combines it with *"pepperoni"* to create *"boo*-beroni."\n\nSince ghosts are often connected to spooky themes and the sound *"boo"* is a playful ghost sound, the punchline humorously suggests that ghosts would order a pizza with a ghostly twist, replacin

In [17]:
# invoke(None, ...) with a checkpoint_id RESUMES the graph from that exact
# saved point instead of starting over -- this is "time travel": you can
# rewind to any earlier state and continue the graph from there.
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f19f0e3-d196-679c-8001-0427b19c02ad"}})

{'topic': 'pizza',
 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni',
 'explanation': 'The joke plays on the word "boo," which is a common sound or exclamation associated with ghosts, and the Italian word "peperoni," which is a type of salami often used as a pizza topping. By combining "boo" and "peperoni," the joke creates a humorous and ghost-themed version of the popular pizza topping, resulting in "booberoni." This kind of wordplay is a common technique in jokes to create humor through unexpected or clever combinations of words.'}

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word "boo," which is a common sound or exclamation associated with ghosts, and the Italian word "peperoni," which is a type of salami often used as a pizza topping. By combining "boo" and "peperoni," the joke creates a humorous and ghost-themed version of the popular pizza topping, resulting in "booberoni." This kind of wordplay is a common technique in jokes to create humor through unexpected or clever combinations of words.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e9-62e7-6f3e-8003-f4c5161a9654'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-23T16:20:42.318391+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e9-5818-6fa2-8002-b78e867f887d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizz

#### Updating State

In [19]:
# update_state lets you REACH INTO a specific checkpoint and change its data
# directly -- here, changing the topic from whatever it was to 'samosa',
# at that EXACT point in history. This creates a NEW checkpoint branching
# off the edited one, without touching the original.
workflow.update_state(
    {"configurable": {"thread_id": "1", "checkpoint_id": "1f19f0e3-d196-679c-8001-0427b19c02ad", "checkpoint_ns": ""}},
    {'topic': 'samosa'}
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19f0ee-7825-6c87-8002-33449ae3e981'}}

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0ee-7825-6c87-8002-33449ae3e981'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-08-23T16:22:58.763454+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0e3-d196-679c-8001-0427b19c02ad'}}, tasks=(PregelTask(id='7ad1c7ce-8547-1ea5-f92f-b44e2d23091f', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word "boo," which is a common sound or exclamation associated with ghosts, and the Italian word "peperoni," which is a type of salami often used as a pizza to

In [21]:
# ⚠️ again, this checkpoint_id must be copied from YOUR OWN Cell 19 output --
# specifically the NEW checkpoint created by the update_state() call above.
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f19f0ee-7825-6c87-8002-33449ae3e981"}})
# the graph re-runs generate_explanation using the EDITED topic ('samosa'),
# producing a joke/explanation for a DIFFERENT topic than the original run --
# proof that state was genuinely rewritten, not just relabeled.

{'topic': 'samosa',
 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni',
 'explanation': 'The joke plays on the word "boo," which is a common sound used to startle or scare someone, often associated with ghosts. The setup leads the listener to expect a silly or spooky answer. By adding "boo" to "beroni," it mimics the word "pepperoni," which is a popular pizza topping. So, the punchline "Boo-beroni" is a pun that combines the ghostly "boo" with "pepperoni," resulting in a humorous and unexpected pizza order from ghosts.'}

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'What kind of pizza do ghosts order?\n*Boo*-beroni', 'explanation': 'The joke plays on the word "boo," which is a common sound used to startle or scare someone, often associated with ghosts. The setup leads the listener to expect a silly or spooky answer. By adding "boo" to "beroni," it mimics the word "pepperoni," which is a popular pizza topping. So, the punchline "Boo-beroni" is a pun that combines the ghostly "boo" with "pepperoni," resulting in a humorous and unexpected pizza order from ghosts.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0f1-7d8c-6d85-8003-79d36a4b808a'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-23T16:24:19.860592+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0ee-7825-6c87-8002-33449ae3e981'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'samosa', 'joke': 'W

### Fault Tolerance

In [23]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [24]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [25]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang -- YOU will interrupt this manually
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [26]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [27]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")
# ACTION REQUIRED: once you see "⏳ Step 2 hanging...", click the STOP/interrupt
# button in the Colab toolbar (or Runtime -> Interrupt execution) to simulate
# a real-world crash mid-workflow.

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [28]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)
# THE KEY PROOF: this does NOT re-run step_1 (you won't see "✅ Step 1 executed"
# again). Because step_1's result was already checkpointed BEFORE the crash,
# the graph resumes from step_2 -- exactly where it left off.


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


KeyboardInterrupt: 

In [29]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0f3-99eb-67e1-8001-d90ae34dcefd'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-23T16:25:16.522473+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0f3-99e6-6a1c-8000-5be91de9a7e0'}}, tasks=(PregelTask(id='c4fe948d-2645-b917-73c6-b9ea3400582f', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f19f0f3-99e6-6a1c-8000-5be91de9a7e0'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-23T16:25:16.520480+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch